In [1]:
import sys
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer
# add original folder into PYTHONPATH
project_root = Path().resolve().parent   
sys.path.append(str(project_root))
from ml.churn_model.features.engineering import engineer_features
from ml.churn_model.data.preprocessing import build_preprocessor

#Load data

DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
#Remove customerID
df = df.drop(columns=["customerID"])

In [3]:
df.shape

(7043, 20)

In [4]:
#Convert TotalCharges
df["TotalCharges"].dtype

<StringDtype(storage='python', na_value=nan)>

In [5]:
#Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)
df["TotalCharges"].isna().sum()

np.int64(11)

In [6]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
#Encode target
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [8]:
#checking info
df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [9]:
#checking columns
df.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [10]:

# Save clean dataset for analytics

PROCESSED_DATA_DIR = Path(
    "../data/processed"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CLEAN_DATA_PATH = (
    PROCESSED_DATA_DIR
    / "customer_churn_clean.csv"
)


# Remove identifier if it still exists
if "customerID" in df.columns:

    df = df.drop(
        columns=["customerID"]
    )


# Validate required analytics columns
required_columns = [
    "tenure",
    "Contract",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if missing_columns:

    raise ValueError(
        "Missing required analytics columns: "
        f"{missing_columns}"
    )


# Save
df.to_csv(
    CLEAN_DATA_PATH,
    index=False,
)


print(
    "Clean analytics dataset saved successfully."
)

print(
    "Path:",
    CLEAN_DATA_PATH.resolve()
)

print(
    "Shape:",
    df.shape
)

print(
    "\nChurn distribution:"
)

print(
    df["Churn"].value_counts()
)

Clean analytics dataset saved successfully.
Path: D:\Research_Project\customer-churn-prediction\data\processed\customer_churn_clean.csv
Shape: (7043, 20)

Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [11]:
#check type Churn save in file
import pandas as pd

df = pd.read_csv(
    "../data/processed/customer_churn_clean.csv"
)

print(df["Churn"].unique())
print(df["Churn"].value_counts())
print(df["Churn"].dtype)

[0 1]
Churn
0    5174
1    1869
Name: count, dtype: int64
int64


In [12]:
# X / y
X = df.drop(columns=["Churn"])

y = df["Churn"]
X.shape

(7043, 19)

In [13]:
y.shape

(7043,)

In [14]:

#Train / Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)



X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)


In [15]:
#check feature engineering
X_train = engineer_features(
    X_train
)

X_test = engineer_features(
    X_test
)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTraining target ratio:")
print(y_train.value_counts(normalize=True))


Training target distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Training target ratio:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64


In [16]:
#check feature engineering
engineered_features = [
    "tenure_group",
    "is_new_customer",
    "is_loyal_customer",
    "avg_monthly_spend",
    "customer_value",
    "is_long_term_contract",
    "automatic_payment",
    "is_electronic_check",
    "has_tech_support",
    "has_online_security",
    "num_support_services",
    "has_support_service",
    "num_services",
    "uses_streaming",
    "has_family",
]

X_train[
    engineered_features
].head()

,tenure_group,is_new_customer,is_loyal_customer,avg_monthly_spend,customer_value,is_long_term_contract,automatic_payment,is_electronic_check,has_tech_support,has_online_security,num_support_services,has_support_service,num_services,uses_streaming,has_family
3738,25-48,0,0,48.618571,1722.00,0,0,1,0,0,1,1,3,1,0
3151,13-24,0,0,76.770000,1126.50,0,0,0,0,1,1,1,2,0,1
4860,13-24,0,0,45.411538,527.15,1,0,0,1,1,3,1,3,0,1
3867,25-48,0,0,73.296154,1911.00,1,1,0,0,0,2,1,5,1,1
3810,0-12,1,0,44.550000,44.55,0,0,1,0,0,0,0,1,0,1


In [17]:
#check tenure_group
X_train[
    "tenure_group"
].value_counts()

tenure_group
49+      1789
0-12     1737
25-48    1290
13-24     818
Name: count, dtype: int64

In [18]:
#check numerical engineered features
X_train[
    [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "avg_monthly_spend",
        "customer_value",
        "num_services",
        "num_support_services",
    ]
].head()

,tenure,MonthlyCharges,TotalCharges,avg_monthly_spend,customer_value,num_services,num_support_services
3738,35,49.20,1701.65,48.618571,1722.00,3,1
3151,15,75.10,1151.55,76.770000,1126.50,2,1
4860,13,40.55,590.35,45.411538,527.15,3,3
3867,26,73.50,1905.70,73.296154,1911.00,5,2
3810,1,44.55,44.55,44.550000,44.55,1,0


In [19]:
#check NaN and Infinity
X_train[
    [
        "avg_monthly_spend",
        "customer_value"
    ]
].isna().sum()

avg_monthly_spend    0
customer_value       0
dtype: int64

In [20]:
np.isinf(
    X_train[
        [
            "avg_monthly_spend",
            "customer_value"
        ]
    ]
).sum()

avg_monthly_spend    0
customer_value       0
dtype: int64

In [21]:
#detect numerical / categorical features again
preprocessor = build_preprocessor(X_train)
# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same preprocessing to test data
X_test_processed = preprocessor.transform(X_test)
print("Numerical features:")
print(X_train_processed)

print("\nCategorical features:")
print(X_test_processed)

Numerical features:
[[-0.44177295  0.10237124 -0.52197565 ...  0.          1.
   0.        ]
 [-0.44177295 -0.71174346  0.33747781 ...  1.          0.
   0.        ]
 [-0.44177295 -0.79315493 -0.80901319 ...  1.          0.
   0.        ]
 ...
 [ 2.2636062  -0.30468611  1.25666162 ...  0.          1.
   0.        ]
 [-0.44177295 -0.34539184 -1.47766135 ...  1.          0.
   0.        ]
 [-0.44177295 -1.07809507 -1.46936546 ...  0.          0.
   0.        ]]

Categorical features:
[[-0.44177295  1.60848343  1.62997635 ...  0.          0.
   1.        ]
 [ 2.2636062  -0.9966836   1.16872526 ...  0.          0.
   0.        ]
 [-0.44177295  0.34660565  0.44532428 ...  0.          1.
   0.        ]
 ...
 [-0.44177295 -1.11880081 -1.46936546 ...  0.          0.
   0.        ]
 [-0.44177295  0.95719167 -1.50088982 ...  0.          0.
   1.        ]
 [-0.44177295  1.60848343  0.18317439 ...  0.          0.
   1.        ]]


In [22]:
feature_names = (
    preprocessor.get_feature_names_out()
)
print(
    "Number of processed features:",
    len(feature_names)
)

print(feature_names)

print("Number of features:", len(feature_names))
print("Train columns:", X_train_processed.shape[1])

Number of processed features: 64
['numerical__SeniorCitizen' 'numerical__tenure'
 'numerical__MonthlyCharges' 'numerical__TotalCharges'
 'numerical__is_new_customer' 'numerical__is_loyal_customer'
 'numerical__avg_monthly_spend' 'numerical__customer_value'
 'numerical__is_long_term_contract' 'numerical__automatic_payment'
 'numerical__is_electronic_check' 'numerical__has_tech_support'
 'numerical__has_online_security' 'numerical__num_support_services'
 'numerical__has_support_service' 'numerical__num_services'
 'numerical__uses_streaming' 'numerical__charges_per_service'
 'numerical__has_family' 'categorical__gender_Female'
 'categorical__gender_Male' 'categorical__Partner_No'
 'categorical__Partner_Yes' 'categorical__Dependents_No'
 'categorical__Dependents_Yes' 'categorical__PhoneService_No'
 'categorical__PhoneService_Yes' 'categorical__MultipleLines_No'
 'categorical__MultipleLines_No phone service'
 'categorical__MultipleLines_Yes' 'categorical__InternetService_DSL'
 'categorical_

In [24]:
#Convert to DataFrame
if hasattr(
    X_train_processed,
    "toarray"
):
    X_train_processed = (
        X_train_processed.toarray()
    )

if hasattr(
    X_test_processed,
    "toarray"
):
    X_test_processed = (
        X_test_processed.toarray()
    )

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
)
#print(type(X_train_processed))
#print(type(X_test_processed))
print("Processed X_train:", X_train_processed.shape)
print("Processed X_test :", X_test_processed.shape)


Processed X_train: (5634, 64)
Processed X_test : (1409, 64)


In [25]:
#export data to a file
from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)
X_train_processed.to_csv(
    PROCESSED_DATA_DIR / "X_train.csv",
    index=False
)

X_test_processed.to_csv(
    PROCESSED_DATA_DIR / "X_test.csv",
    index=False
)

y_train.to_csv(
    PROCESSED_DATA_DIR / "y_train.csv",
    index=False
)

y_test.to_csv(
    PROCESSED_DATA_DIR / "y_test.csv",
    index=False
)

print("Saved processed datasets successfully.")

print("X_train:", X_train_processed.shape)
print("X_test :", X_test_processed.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Saved processed datasets successfully.
X_train: (5634, 64)
X_test : (1409, 64)
y_train: (5634,)
y_test : (1409,)
